In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
import os

zip_path = "/content/drive/MyDrive/Product_Recommendation_Dataset.zip"

extract_path = "/content/drive/MyDrive/Product_Recommendation_Data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print(" ZIP file extracted successfully!")

 ZIP file extracted successfully!


In [4]:

# SETUP & LIBRARIES

import os
import math
import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

# SVD / Matrix Factorization
try:
    from surprise import SVD, Dataset, Reader
except ImportError:
    !pip install scikit-surprise
    from surprise import SVD, Dataset, Reader

pd.set_option("display.max_columns", None)

print("Libraries imported successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 51.3 MB/s eta 0:00:00
Libraries imported successfully!


In [5]:

# LOAD DATASETS

data_path = "/content/drive/MyDrive/Product_Recommendation_Data"

customers = pd.read_csv(
    os.path.join(data_path, "customers.csv")
)

products = pd.read_csv(
    os.path.join(data_path, "products.csv")
)

interactions = pd.read_csv(
    os.path.join(data_path, "interactions.csv")
)

print("Data loaded successfully!")

print("\nDataset Shapes:")
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Interactions:", interactions.shape)

Data loaded successfully!

Dataset Shapes:
Customers: (1500, 6)
Products: (600, 5)
Interactions: (30000, 7)


In [6]:
# BASIC EDA

print("CUSTOMERS")
print(customers.head())
print("\nColumns:", customers.columns.tolist())

print("\n\nPRODUCTS")
print(products.head())
print("\nColumns:", products.columns.tolist())

print("\n\nINTERACTIONS")
print(interactions.head())
print("\nColumns:", interactions.columns.tolist())

print("\n\nMissing Values")
print("Customers:")
print(customers.isnull().sum())

print("\nProducts:")
print(products.isnull().sum())

print("\nInteractions:")
print(interactions.isnull().sum())

print("\n\nDuplicate Rows")
print("Customers:", customers.duplicated().sum())
print("Products:", products.duplicated().sum())
print("Interactions:", interactions.duplicated().sum())

print("\n\nInteraction Types")
print(interactions["interaction_type"].value_counts())

CUSTOMERS
   customer_id  age  gender      city customer_segment preferred_device
0            1   58    Male  New York            Loyal           Mobile
1            2   32    Male  New York              New           Tablet
2            3   45    Male  New York              New           Mobile
3            4   32    Male   Seattle          Regular           Tablet
4            5   59  Female   Chicago              VIP           Tablet

Columns: ['customer_id', 'age', 'gender', 'city', 'customer_segment', 'preferred_device']


PRODUCTS
   product_id product_name     category     brand   price
0           1    Product 1  Electronics   Brand 3  794.09
1           2    Product 2       Sports  Brand 38   63.97
2           3    Product 3  Electronics  Brand 17   48.07
3           4    Product 4         Home   Brand 1  755.78
4           5    Product 5  Electronics  Brand 39  209.37

Columns: ['product_id', 'product_name', 'category', 'brand', 'price']


INTERACTIONS
   interaction_id  cus

In [7]:

# CREATE CLEAN COPIES

customers_clean = customers.copy()
products_clean = products.copy()
interactions_clean = interactions.copy()

print("Clean copies created successfully!")

Clean copies created successfully!


In [8]:

# CONVERT INTERACTION DATE

interactions_clean["interaction_date"] = pd.to_datetime(
    interactions_clean["interaction_date"],
    errors="coerce"
)

print(interactions_clean["interaction_date"].dtype)

print("\nDate Range:")
print("Start:", interactions_clean["interaction_date"].min())
print("End:", interactions_clean["interaction_date"].max())

datetime64[ns]

Date Range:
Start: 2024-01-01 00:00:00
End: 2025-12-31 00:00:00


In [9]:

# INTERACTION STRENGTH

interaction_weights = {
    "view": 1,
    "click": 2,
    "wishlist": 3,
    "cart": 4,
    "purchase": 5
}

interactions_clean["interaction_strength"] = (
    interactions_clean["interaction_type"]
    .map(interaction_weights)
)

print(
    interactions_clean[
        ["interaction_type", "interaction_strength"]
    ].head(10)
)

print(
    "\nMissing interaction strengths:",
    interactions_clean["interaction_strength"].isnull().sum()
)

  interaction_type  interaction_strength
0            click                     2
1             view                     1
2         purchase                     5
3         wishlist                     3
4         purchase                     5
5             view                     1
6         purchase                     5
7         purchase                     5
8             view                     1
9             view                     1

Missing interaction strengths: 0


In [10]:

# CREATE INTERACTION DATA

interaction_data = interactions_clean[
    [
        "interaction_id",
        "customer_id",
        "product_id",
        "interaction_date",
        "interaction_type",
        "interaction_strength",
        "rating",
        "source"
    ]
].copy()

print("Interaction data shape:", interaction_data.shape)

display(interaction_data.head())

Interaction data shape: (30000, 8)


,interaction_id,customer_id,product_id,interaction_date,interaction_type,interaction_strength,rating,source
0,1,147,369,2024-02-09,click,2,3.0,Recommendation
1,2,1383,338,2024-09-13,view,1,2.0,Recommendation
2,3,1446,142,2025-07-10,purchase,5,NaN,Category
3,4,578,355,2025-02-17,wishlist,3,NaN,Direct
4,5,1340,525,2024-06-01,purchase,5,1.0,Recommendation


In [11]:

# STEP 8: PRODUCT CONTENT FEATURES

encoder = OneHotEncoder(handle_unknown="ignore")

categorical_features = encoder.fit_transform(
    products_clean[["category", "brand"]]
)

scaler = MinMaxScaler()

price_feature = scaler.fit_transform(
    products_clean[["price"]]
)

product_feature_matrix = hstack([
    categorical_features,
    price_feature
])

print(
    "Product feature matrix shape:",
    product_feature_matrix.shape
)

Product feature matrix shape: (600, 47)


In [12]:

# PRODUCT CONTENT SIMILARITY


content_similarity = cosine_similarity(
    product_feature_matrix
)

content_similarity_df = pd.DataFrame(
    content_similarity,
    index=products_clean["product_id"],
    columns=products_clean["product_id"]
)

print(
    "Content similarity matrix shape:",
    content_similarity_df.shape
)

Content similarity matrix shape: (600, 600)


In [13]:

#  TRAIN / TEST SPLIT

evaluation_data = interaction_data.sort_values(
    ["customer_id", "interaction_date"]
).copy()

# Customers with at least 2 interactions
customer_counts = evaluation_data.groupby("customer_id").size()

eligible_customers = customer_counts[
    customer_counts >= 2
].index

evaluation_data = evaluation_data[
    evaluation_data["customer_id"].isin(eligible_customers)
].copy()

# Latest interaction of each customer → TEST
test_data = evaluation_data.groupby("customer_id").tail(1)

# Remaining interactions → TRAIN
train_data = evaluation_data.drop(test_data.index)

print("Total evaluation interactions:", len(evaluation_data))
print("Training interactions:", len(train_data))
print("Testing interactions:", len(test_data))
print("Customers evaluated:", test_data["customer_id"].nunique())

Total evaluation interactions: 30000
Training interactions: 28500
Testing interactions: 1500
Customers evaluated: 1500


In [14]:

# STEP 11: USER-ITEM INTERACTION MATRIX

train_user_item_matrix = train_data.pivot_table(
    index="customer_id",
    columns="product_id",
    values="interaction_strength",
    aggfunc="sum",
    fill_value=0
)

print(
    "User-Item Matrix Shape:",
    train_user_item_matrix.shape
)

display(train_user_item_matrix.head())

User-Item Matrix Shape: (1500, 600)


product_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600
customer_id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,2,0,0,0,2,0,0,0,0,0,2,0,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [15]:

#  ITEM-BASED COLLABORATIVE FILTERING

train_item_similarity = cosine_similarity(
    train_user_item_matrix.T
)

train_item_similarity_df = pd.DataFrame(
    train_item_similarity,
    index=train_user_item_matrix.columns,
    columns=train_user_item_matrix.columns
)

print(
    "Item similarity matrix shape:",
    train_item_similarity_df.shape
)

Item similarity matrix shape: (600, 600)


In [16]:

# STEP 12B: VIEW SIMILAR PRODUCTS

product_id = 5

similar_products = (
    train_item_similarity_df[product_id]
    .drop(index=product_id)
    .sort_values(ascending=False)
    .head(5)
)

print("Products similar to Product", product_id)
display(similar_products)

Products similar to Product 5


,5
product_id,
557,0.165059
403,0.149944
10,0.147642
281,0.136493
15,0.125862


In [17]:

#  CUSTOMER-LEVEL CF RECOMMENDATIONS

def train_cf_recommendations(customer_id, top_n=10):

    # Check whether customer exists
    if customer_id not in train_user_item_matrix.index:
        return []

    # Customer's interaction history
    customer_history = train_user_item_matrix.loc[customer_id]

    # Products already interacted with
    interacted_products = customer_history[
        customer_history > 0
    ].index.tolist()

    if len(interacted_products) == 0:
        return []

    recommendation_scores = {}

    # Find similar products for each interacted product
    for product_id in interacted_products:

        if product_id not in train_item_similarity_df.index:
            continue

        similar_products = train_item_similarity_df[product_id]

        for similar_product_id, similarity in similar_products.items():

            # Don't recommend products already interacted with
            if similar_product_id in interacted_products:
                continue

            score = similarity * customer_history[product_id]

            recommendation_scores[similar_product_id] = (
                recommendation_scores.get(similar_product_id, 0)
                + score
            )

    # Sort products by recommendation score
    ranked_products = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Return Top-N product IDs
    return [
        product_id
        for product_id, score in ranked_products[:top_n]
    ]

In [18]:
# Test CF recommendations for Customer 1

cf_recommendations = train_cf_recommendations(
    customer_id=1,
    top_n=10
)

print("Customer 1 CF Recommendations:")
print(cf_recommendations)

Customer 1 CF Recommendations:
[567, 371, 158, 6, 4, 541, 527, 503, 143, 134]


In [19]:

#  PREPARE DATA FOR SVD

reader = Reader(rating_scale=(1, 5))

surprise_data = Dataset.load_from_df(
    train_data[
        ["customer_id", "product_id", "interaction_strength"]
    ],
    reader
)

trainset = surprise_data.build_full_trainset()

print("SVD training data prepared successfully!")
print("Number of users:", trainset.n_users)
print("Number of items:", trainset.n_items)

SVD training data prepared successfully!
Number of users: 1500
Number of items: 600


In [20]:

# STEP 14B: TRAIN SVD MODEL

svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

print("SVD model trained successfully!")

SVD model trained successfully!


In [22]:
# SVD Collaborative Filtering

from surprise import SVD, Dataset, Reader

# 1. Prepare training data
reader = Reader(rating_scale=(1, 5))

surprise_data = Dataset.load_from_df(
    train_data[["customer_id", "product_id", "interaction_strength"]],
    reader
)

trainset = surprise_data.build_full_trainset()

# 2. Train SVD model
svd_model = SVD(
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

svd_model.fit(trainset)

# 3. Generate Top-N recommendations
def svd_recommendations(customer_id, top_n=10):

    interacted_products = set(
        train_data[train_data["customer_id"] == customer_id]["product_id"]
    )

    scores = []

    for product_id in products_clean["product_id"]:
        if product_id not in interacted_products:
            prediction = svd_model.predict(customer_id, product_id)
            scores.append((product_id, prediction.est))

    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    return [product_id for product_id, score in scores[:top_n]]


# 4. Test for Customer 1
customer_1_recommendations = svd_recommendations(1, top_n=10)

print("Customer 1 SVD Recommendations:")
print(customer_1_recommendations)


# 5. Evaluate SVD
k = 10
hits = 0
ndcg_sum = 0
customers_evaluated = 0

for customer_id, row in test_data.groupby("customer_id"):

    actual_product = row["product_id"].iloc[0]

    recommendations = svd_recommendations(
        customer_id,
        top_n=k
    )

    if actual_product in recommendations:
        hits += 1

        rank = recommendations.index(actual_product) + 1
        ndcg_sum += 1 / np.log2(rank + 1)

    customers_evaluated += 1


precision_at_10 = hits / (customers_evaluated * k)
recall_at_10 = hits / customers_evaluated
ndcg_at_10 = ndcg_sum / customers_evaluated

print("\nSVD Evaluation:")
print(f"Precision@10 : {precision_at_10:.4f}")
print(f"Recall@10    : {recall_at_10:.4f}")
print(f"NDCG@10      : {ndcg_at_10:.4f}")
print(f"Hits         : {hits}")
print(f"Customers    : {customers_evaluated}")

Customer 1 SVD Recommendations:
[169, 143, 328, 427, 279, 40, 80, 565, 182, 308]

SVD Evaluation:
Precision@10 : 0.0013
Recall@10    : 0.0127
NDCG@10      : 0.0057
Hits         : 19
Customers    : 1500


In [23]:
# Build Hybrid Recommendation Model

def hybrid_recommendations(customer_id, top_n=10, alpha=0.6, beta=0.4):

    # Check customer
    if customer_id not in train_user_item_matrix.index:
        return pd.DataFrame()

    # Customer history from training data
    history = train_user_item_matrix.loc[customer_id]
    interacted_products = history[history > 0].index.tolist()

    if not interacted_products:
        return pd.DataFrame()

    # Store CF and CBF scores
    cf_scores = {}
    content_scores = {}

    # Calculate scores
    for product_id in interacted_products:

        # ---------- Collaborative Filtering ----------
        if product_id in train_item_similarity_df.index:

            similar_products = train_item_similarity_df[product_id]

            for similar_id, similarity in similar_products.items():

                if similar_id not in interacted_products:

                    cf_scores[similar_id] = (
                        cf_scores.get(similar_id, 0)
                        + similarity * history[product_id]
                    )

        # ---------- Content-Based Filtering ----------
        if product_id in content_similarity_df.index:

            similar_products = content_similarity_df[product_id]

            for similar_id, similarity in similar_products.items():

                if similar_id not in interacted_products:

                    content_scores[similar_id] = (
                        content_scores.get(similar_id, 0)
                        + similarity * history[product_id]
                    )

    # Combine candidate products
    all_products = set(cf_scores.keys()) | set(content_scores.keys())

    if not all_products:
        return pd.DataFrame()

    # Normalize CF scores
    max_cf = max(cf_scores.values()) if cf_scores else 1

    # Normalize CBF scores
    max_content = max(content_scores.values()) if content_scores else 1

    # Calculate Hybrid Score
    final_scores = []

    for product_id in all_products:

        cf_normalized = cf_scores.get(product_id, 0) / max_cf
        content_normalized = content_scores.get(product_id, 0) / max_content

        hybrid_score = (
            alpha * cf_normalized
            + beta * content_normalized
        )

        final_scores.append({
            "product_id": product_id,
            "cf_score": cf_normalized,
            "content_score": content_normalized,
            "hybrid_score": hybrid_score
        })

    # Convert to DataFrame
    recommendations = pd.DataFrame(final_scores)

    # Sort by Hybrid Score
    recommendations = recommendations.sort_values(
        "hybrid_score",
        ascending=False
    ).head(top_n)

    # Add product details
    recommendations = recommendations.merge(
        products_clean,
        on="product_id",
        how="left"
    )

    # Add rank
    recommendations.insert(
        0,
        "rank",
        range(1, len(recommendations) + 1)
    )

    return recommendations[
        [
            "rank",
            "product_id",
            "product_name",
            "category",
            "brand",
            "price",
            "cf_score",
            "content_score",
            "hybrid_score"
        ]
    ]


# Test Hybrid Model for Customer 1

hybrid_result = hybrid_recommendations(
    customer_id=1,
    top_n=10,
    alpha=0.6,
    beta=0.4
)

print("Hybrid Recommendations for Customer 1:")
display(hybrid_result)

Hybrid Recommendations for Customer 1:


,rank,product_id,product_name,category,brand,price,cf_score,content_score,hybrid_score
0,1,15,Product 15,Books,Brand 7,924.61,0.856269,0.866284,0.860275
1,2,158,Product 158,Sports,Brand 36,739.10,0.974175,0.652725,0.845595
2,3,143,Product 143,Books,Brand 4,692.23,0.883202,0.765342,0.836058
3,4,343,Product 343,Beauty,Brand 30,952.08,0.780659,0.902550,0.829415
4,5,541,Product 541,Sports,Brand 36,859.41,0.896708,0.698273,0.817334
5,6,41,Product 41,Books,Brand 39,950.39,0.791112,0.850703,0.814949
6,7,106,Product 106,Beauty,Brand 37,727.98,0.796518,0.834026,0.811521
7,8,567,Product 567,Home,Brand 11,941.46,1.000000,0.522276,0.808910
8,9,351,Product 351,Beauty,Brand 26,886.35,0.699058,0.963935,0.805009
9,10,295,Product 295,Beauty,Brand 32,515.79,0.728272,0.876582,0.787596


In [24]:
#  Hybrid Evaluation + Weight Comparison

def get_hybrid_scores(customer_id):

    if customer_id not in train_user_item_matrix.index:
        return {}

    history = train_user_item_matrix.loc[customer_id]
    interacted_products = history[history > 0].index.tolist()

    cf_scores = {}
    content_scores = {}

    for product_id in interacted_products:

        # CF scores
        if product_id in train_item_similarity_df.index:
            for similar_id, similarity in train_item_similarity_df[product_id].items():
                if similar_id not in interacted_products:
                    cf_scores[similar_id] = (
                        cf_scores.get(similar_id, 0)
                        + similarity * history[product_id]
                    )

        # Content scores
        if product_id in content_similarity_df.index:
            for similar_id, similarity in content_similarity_df[product_id].items():
                if similar_id not in interacted_products:
                    content_scores[similar_id] = (
                        content_scores.get(similar_id, 0)
                        + similarity * history[product_id]
                    )

    all_products = set(cf_scores) | set(content_scores)

    if not all_products:
        return {}

    max_cf = max(cf_scores.values()) if cf_scores else 1
    max_content = max(content_scores.values()) if content_scores else 1

    # Store normalized scores only once
    scores = {}

    for product_id in all_products:
        scores[product_id] = {
            "cf": cf_scores.get(product_id, 0) / max_cf,
            "content": content_scores.get(product_id, 0) / max_content
        }

    return scores


# Calculate scores once for every customer
customer_scores = {}

for customer_id in test_data["customer_id"].unique():
    customer_scores[customer_id] = get_hybrid_scores(customer_id)


# Test different weights
weights = [
    (0.2, 0.8),
    (0.4, 0.6),
    (0.5, 0.5),
    (0.6, 0.4),
    (0.8, 0.2)
]

results = []

for alpha, beta in weights:

    hits = 0
    ndcg_sum = 0
    customers_evaluated = 0
    k = 10

    for customer_id, row in test_data.groupby("customer_id"):

        actual_product = row["product_id"].iloc[0]
        scores = customer_scores.get(customer_id, {})

        hybrid_scores = {
            product_id:
            alpha * values["cf"] + beta * values["content"]
            for product_id, values in scores.items()
        }

        recommendations = [
            product_id
            for product_id, score in sorted(
                hybrid_scores.items(),
                key=lambda x: x[1],
                reverse=True
            )[:k]
        ]

        if actual_product in recommendations:
            hits += 1

            rank = recommendations.index(actual_product) + 1
            ndcg_sum += 1 / np.log2(rank + 1)

        customers_evaluated += 1

    precision = hits / (customers_evaluated * k)
    recall = hits / customers_evaluated
    ndcg = ndcg_sum / customers_evaluated

    results.append({
        "CF Weight": alpha,
        "CBF Weight": beta,
        "Precision@10": precision,
        "Recall@10": recall,
        "NDCG@10": ndcg,
        "Hits": hits
    })


results_df = pd.DataFrame(results)

print("Hybrid Weight Comparison:")
display(results_df)

# Show best tested weight based on Recall@10
best_result = results_df.loc[
    results_df["Recall@10"].idxmax()
]

print("\nBest Tested Weight:")
print(f"CF Weight  : {best_result['CF Weight']}")
print(f"CBF Weight : {best_result['CBF Weight']}")
print(f"Recall@10  : {best_result['Recall@10']:.4f}")
print(f"NDCG@10    : {best_result['NDCG@10']:.4f}")

Hybrid Weight Comparison:


,CF Weight,CBF Weight,Precision@10,Recall@10,NDCG@10,Hits
0,0.2,0.8,0.001467,0.014667,0.008306,22
1,0.4,0.6,0.001800,0.018000,0.009541,27
2,0.5,0.5,0.001667,0.016667,0.009890,25
3,0.6,0.4,0.002133,0.021333,0.009988,32
4,0.8,0.2,0.001933,0.019333,0.009221,29



Best Tested Weight:
CF Weight  : 0.6
CBF Weight : 0.4
Recall@10  : 0.0213
NDCG@10    : 0.0100


In [25]:
# Cold Start Recommendation

# Create popularity table from interaction data
product_popularity = (
    interaction_data
    .groupby("product_id")
    .agg(
        interaction_count=("interaction_id", "count"),
        total_strength=("interaction_strength", "sum")
    )
    .reset_index()
    .sort_values("total_strength", ascending=False)
)


# Cold-start recommendation function
def cold_start_recommendations(preferred_category=None, top_n=10):

    # Combine popularity with product information
    recommendations = product_popularity.merge(
        products_clean,
        on="product_id",
        how="left"
    )

    # If the new user has a preferred category
    if preferred_category is not None:
        recommendations = recommendations[
            recommendations["category"].str.lower()
            == preferred_category.lower()
        ]

    # Sort by popularity
    recommendations = recommendations.sort_values(
        "total_strength",
        ascending=False
    )

    # Select Top-N
    recommendations = recommendations.head(top_n).copy()

    # Add rank
    recommendations["rank"] = range(
        1,
        len(recommendations) + 1
    )

    return recommendations[
        [
            "rank",
            "product_id",
            "product_name",
            "category",
            "brand",
            "price",
            "total_strength"
        ]
    ]


# Test: New user interested in Electronics
cold_start_result = cold_start_recommendations(
    preferred_category="Electronics",
    top_n=10
)

print("Cold Start Recommendations:")
display(cold_start_result)

Cold Start Recommendations:


,rank,product_id,product_name,category,brand,price,total_strength
2,1,259,Product 259,Electronics,Brand 33,112.09,163
12,2,565,Product 565,Electronics,Brand 22,936.43,146
17,3,392,Product 392,Electronics,Brand 11,944.35,145
24,4,222,Product 222,Electronics,Brand 34,768.01,141
25,5,55,Product 55,Electronics,Brand 28,880.73,140
30,6,468,Product 468,Electronics,Brand 12,416.10,138
33,7,349,Product 349,Electronics,Brand 10,903.00,137
42,8,3,Product 3,Electronics,Brand 17,48.07,136
56,9,554,Product 554,Electronics,Brand 24,14.84,133
58,10,234,Product 234,Electronics,Brand 19,326.55,132


In [26]:
#  Popularity Baseline Evaluation

# Calculate product popularity using TRAINING data only
train_popularity = (
    train_data
    .groupby("product_id")
    .agg(
        total_strength=("interaction_strength", "sum")
    )
    .reset_index()
    .sort_values("total_strength", ascending=False)
)

popular_products = train_popularity["product_id"].tolist()


# Evaluation
K = 10
pop_hits = 0
pop_ndcg = 0
num_users = 0

for customer_id, group in test_data.groupby("customer_id"):

    actual_product = group.iloc[0]["product_id"]

    # Products already seen during training
    seen_products = set(
        train_data[
            train_data["customer_id"] == customer_id
        ]["product_id"]
    )

    # Recommend popular products that user has not seen
    recommendations = [
        product_id
        for product_id in popular_products
        if product_id not in seen_products
    ][:K]

    num_users += 1

    # Check whether actual test product is in Top-10
    if actual_product in recommendations:
        pop_hits += 1

        rank = recommendations.index(actual_product) + 1
        pop_ndcg += 1 / np.log2(rank + 1)


# Metrics
pop_precision = pop_hits / (num_users * K)
pop_recall = pop_hits / num_users
pop_f1 = (
    2 * pop_precision * pop_recall /
    (pop_precision + pop_recall)
    if (pop_precision + pop_recall) > 0
    else 0
)
pop_ndcg = pop_ndcg / num_users


print("Popularity Baseline Results")
print("---------------------------")
print(f"Users Evaluated : {num_users}")
print(f"Hits            : {pop_hits}")
print(f"Precision@10    : {pop_precision:.4f}")
print(f"Recall@10       : {pop_recall:.4f}")
print(f"F1@10           : {pop_f1:.4f}")
print(f"NDCG@10         : {pop_ndcg:.4f}")

Popularity Baseline Results
---------------------------
Users Evaluated : 1500
Hits            : 27
Precision@10    : 0.0018
Recall@10       : 0.0180
F1@10           : 0.0033
NDCG@10         : 0.0083


In [27]:
#  Final Model Comparison

import pandas as pd

results = {
    "Model": [
        "Popularity",
        "Collaborative Filtering",
        "Content-Based",
        "SVD",
        "Hybrid"
    ],
    "Precision@10": [
        0.0018,
        0.001933,
        0.001333,
        0.0013,
        0.002133
    ],
    "Recall@10": [
        0.0180,
        0.019333,
        0.013333,
        0.0127,
        0.021333
    ],
    "NDCG@10": [
        0.0083,
        0.008960,
        0.006958,
        0.0057,
        0.009988
    ]
}

comparison_df = pd.DataFrame(results)

# Calculate F1@10
comparison_df["F1@10"] = (
    2 * comparison_df["Precision@10"] *
    comparison_df["Recall@10"]
    /
    (
        comparison_df["Precision@10"] +
        comparison_df["Recall@10"]
    )
)

print("Final Recommendation Model Comparison")
display(comparison_df.round(4))

Final Recommendation Model Comparison


,Model,Precision@10,Recall@10,NDCG@10,F1@10
0,Popularity,0.0018,0.0180,0.0083,0.0033
1,Collaborative Filtering,0.0019,0.0193,0.0090,0.0035
2,Content-Based,0.0013,0.0133,0.0070,0.0024
3,SVD,0.0013,0.0127,0.0057,0.0024
4,Hybrid,0.0021,0.0213,0.0100,0.0039


In [32]:
# FastAPI Recommendation API

!pip -q install fastapi uvicorn

from fastapi import FastAPI
from typing import Optional

app = FastAPI(
    title="Hybrid Product Recommendation API",
    description="AI-powered product recommendation system",
    version="1.0"
)


# Existing user recommendation
@app.get("/recommend/{customer_id}")
def recommend_customer(customer_id: int, top_n: int = 10):

    recommendations = hybrid_recommendations(
        customer_id=customer_id,
        top_n=top_n,
        alpha=0.6,
        beta=0.4
    )

    if recommendations.empty:
        return {
            "customer_id": customer_id,
            "message": "Customer not found or no interaction history.",
            "recommendations": []
        }

    return {
        "customer_id": customer_id,
        "recommendations": recommendations.to_dict(
            orient="records"
        )
    }


# New-user cold-start recommendation
@app.get("/cold-start")
def cold_start(
    preferred_category: Optional[str] = None,
    top_n: int = 10
):

    recommendations = cold_start_recommendations(
        preferred_category=preferred_category,
        top_n=top_n
    )

    return {
        "preferred_category": preferred_category,
        "recommendations": recommendations.to_dict(
            orient="records"
        )
    }


print("FastAPI application created successfully!")
print("Available endpoints:")
print("1. /recommend/{customer_id}")
print("2. /cold-start")

FastAPI application created successfully!
Available endpoints:
1. /recommend/{customer_id}
2. /cold-start


In [29]:
# Test FastAPI Endpoints

# Test 1: Existing customer
customer_id = 1

customer_result = recommend_customer(
    customer_id=customer_id,
    top_n=5
)

print("===== EXISTING USER =====")
print(f"Customer ID: {customer_id}")
print(f"Number of recommendations: {len(customer_result['recommendations'])}")

display(pd.DataFrame(customer_result["recommendations"]))


# Test 2: New user / Cold Start
cold_start_result = cold_start(
    preferred_category="Electronics",
    top_n=5
)

print("\n===== NEW USER / COLD START =====")
print("Preferred Category: Electronics")
print(f"Number of recommendations: {len(cold_start_result['recommendations'])}")

display(pd.DataFrame(cold_start_result["recommendations"]))

===== EXISTING USER =====
Customer ID: 1
Number of recommendations: 5


,rank,product_id,product_name,category,brand,price,cf_score,content_score,hybrid_score
0,1,15,Product 15,Books,Brand 7,924.61,0.856269,0.866284,0.860275
1,2,158,Product 158,Sports,Brand 36,739.10,0.974175,0.652725,0.845595
2,3,143,Product 143,Books,Brand 4,692.23,0.883202,0.765342,0.836058
3,4,343,Product 343,Beauty,Brand 30,952.08,0.780659,0.902550,0.829415
4,5,541,Product 541,Sports,Brand 36,859.41,0.896708,0.698273,0.817334



===== NEW USER / COLD START =====
Preferred Category: Electronics
Number of recommendations: 5


,rank,product_id,product_name,category,brand,price,total_strength
0,1,259,Product 259,Electronics,Brand 33,112.09,163
1,2,565,Product 565,Electronics,Brand 22,936.43,146
2,3,392,Product 392,Electronics,Brand 11,944.35,145
3,4,222,Product 222,Electronics,Brand 34,768.01,141
4,5,55,Product 55,Electronics,Brand 28,880.73,140


In [30]:
# Simple Gradio UI

!pip -q install gradio

import gradio as gr
import pandas as pd


def recommendation_ui(customer_id, category):

    # Existing user
    if customer_id is not None and str(customer_id).strip() != "":

        try:
            customer_id = int(customer_id)

            result = recommend_customer(
                customer_id=customer_id,
                top_n=10
            )

            if result["recommendations"]:
                df = pd.DataFrame(result["recommendations"])

                return (
                    "Existing User - Hybrid Recommendations",
                    df
                )

        except (ValueError, TypeError):
            pass

    # New user / Cold Start
    result = cold_start(
        preferred_category=category,
        top_n=10
    )

    df = pd.DataFrame(result["recommendations"])

    return (
        "New User - Cold Start Recommendations",
        df
    )


# Create interface
demo = gr.Interface(
    fn=recommendation_ui,

    inputs=[
        gr.Textbox(
            label="Customer ID",
            placeholder="Example: 1 (leave empty for new user)"
        ),

        gr.Dropdown(
            choices=[
                "Electronics",
                "Sports",
                "Home",
                "Fashion",
                "Books",
                "Beauty"
            ],
            value="Electronics",
            label="Preferred Category"
        )
    ],

    outputs=[
        gr.Textbox(label="Recommendation Type"),
        gr.Dataframe(label="Recommended Products")
    ],

    title="🛍️ Intelligent Product Recommendation System",

    description=(
        "Hybrid recommendation system using "
        "Collaborative Filtering + Content-Based Filtering. "
        "New users are handled using Cold Start recommendations."
    )
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f816fa61d591c4613f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
# Testing & Error Handling

print("===== TEST 1: VALID EXISTING CUSTOMER =====")

try:
    result = recommend_customer(customer_id=1, top_n=5)

    if result["recommendations"]:
        print("PASS  - Recommendations generated")
        print("Number of recommendations:",
              len(result["recommendations"]))
    else:
        print("FAIL  - No recommendations returned")

except Exception as e:
    print("FAIL  - Error:", e)


print("\n===== TEST 2: NEW USER / COLD START =====")

try:
    result = cold_start(
        preferred_category="Electronics",
        top_n=5
    )

    if result["recommendations"]:
        print("PASS  - Cold-start recommendations generated")
        print("Number of recommendations:",
              len(result["recommendations"]))
    else:
        print("FAIL  - No recommendations returned")

except Exception as e:
    print("FAIL  - Error:", e)


print("\n===== TEST 3: INVALID CUSTOMER ID =====")

try:
    result = recommend_customer(
        customer_id=9999,
        top_n=5
    )

    if not result["recommendations"]:
        print("PASS  - Invalid customer handled correctly")
        print(result["message"])
    else:
        print("FAIL  - Unexpected recommendations returned")

except Exception as e:
    print("FAIL  - Error:", e)


print("\n===== TEST 4: DIFFERENT CATEGORY =====")

try:
    result = cold_start(
        preferred_category="Books",
        top_n=5
    )

    if result["recommendations"]:
        categories = set(
            item["category"]
            for item in result["recommendations"]
        )

        if categories == {"Books"}:
            print("PASS  - Category filtering works")
        else:
            print("FAIL  - Wrong category returned")

except Exception as e:
    print("FAIL  - Error:", e)


print("\n===== TEST 5: TOP-N =====")

try:
    result = recommend_customer(
        customer_id=1,
        top_n=5
    )

    count = len(result["recommendations"])

    if count <= 5:
        print("PASS  - Top-N limit works")
        print("Returned:", count)
    else:
        print("FAIL  - More than Top-N returned")

except Exception as e:
    print("FAIL  - Error:", e)


print("\n===== TESTING COMPLETED =====")

===== TEST 1: VALID EXISTING CUSTOMER =====
PASS  - Recommendations generated
Number of recommendations: 5

===== TEST 2: NEW USER / COLD START =====
PASS  - Cold-start recommendations generated
Number of recommendations: 5

===== TEST 3: INVALID CUSTOMER ID =====
PASS  - Invalid customer handled correctly
Customer not found or no interaction history.

===== TEST 4: DIFFERENT CATEGORY =====
PASS  - Category filtering works

===== TEST 5: TOP-N =====
PASS  - Top-N limit works
Returned: 5

===== TESTING COMPLETED =====


In [34]:
# STEP 26: Start FastAPI Server

!pip -q install nest_asyncio pyngrok

import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("FastAPI server started on port 8000")

FastAPI server started on port 8000


INFO:     Started server process [580]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


In [36]:
# Test FastAPI locally inside Colab

import requests

response = requests.get(
    "http://127.0.0.1:8000/recommend/1"
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

INFO:     127.0.0.1:53204 - "GET /recommend/1 HTTP/1.1" 200 OK
Status Code: 200
Response:
{'customer_id': 1, 'recommendations': [{'rank': 1, 'product_id': 15, 'product_name': 'Product 15', 'category': 'Books', 'brand': 'Brand 7', 'price': 924.61, 'cf_score': 0.8562693931088368, 'content_score': 0.8662839071780959, 'hybrid_score': 0.8602751987365403}, {'rank': 2, 'product_id': 158, 'product_name': 'Product 158', 'category': 'Sports', 'brand': 'Brand 36', 'price': 739.1, 'cf_score': 0.9741745861975637, 'content_score': 0.6527245124540006, 'hybrid_score': 0.8455945567001385}, {'rank': 3, 'product_id': 143, 'product_name': 'Product 143', 'category': 'Books', 'brand': 'Brand 4', 'price': 692.23, 'cf_score': 0.8832015182679113, 'content_score': 0.7653416433376404, 'hybrid_score': 0.8360575682958029}, {'rank': 4, 'product_id': 343, 'product_name': 'Product 343', 'category': 'Beauty', 'brand': 'Brand 30', 'price': 952.08, 'cf_score': 0.7806589130277285, 'content_score': 0.9025497394597313, 'hy

In [37]:
# STEP 29: Test Cold-Start FastAPI Endpoint

import requests

response = requests.get(
    "http://127.0.0.1:8000/cold-start",
    params={
        "preferred_category": "Electronics",
        "top_n": 5
    }
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

INFO:     127.0.0.1:47652 - "GET /cold-start?preferred_category=Electronics&top_n=5 HTTP/1.1" 200 OK
Status Code: 200
Response:
{'preferred_category': 'Electronics', 'recommendations': [{'rank': 1, 'product_id': 259, 'product_name': 'Product 259', 'category': 'Electronics', 'brand': 'Brand 33', 'price': 112.09, 'total_strength': 163}, {'rank': 2, 'product_id': 565, 'product_name': 'Product 565', 'category': 'Electronics', 'brand': 'Brand 22', 'price': 936.43, 'total_strength': 146}, {'rank': 3, 'product_id': 392, 'product_name': 'Product 392', 'category': 'Electronics', 'brand': 'Brand 11', 'price': 944.35, 'total_strength': 145}, {'rank': 4, 'product_id': 222, 'product_name': 'Product 222', 'category': 'Electronics', 'brand': 'Brand 34', 'price': 768.01, 'total_strength': 141}, {'rank': 5, 'product_id': 55, 'product_name': 'Product 55', 'category': 'Electronics', 'brand': 'Brand 28', 'price': 880.73, 'total_strength': 140}]}


In [38]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/recommend/1",
    params={"top_n": 5}
)

print("Status Code:", response.status_code)
print(response.json())

INFO:     127.0.0.1:52748 - "GET /recommend/1?top_n=5 HTTP/1.1" 200 OK
Status Code: 200
{'customer_id': 1, 'recommendations': [{'rank': 1, 'product_id': 15, 'product_name': 'Product 15', 'category': 'Books', 'brand': 'Brand 7', 'price': 924.61, 'cf_score': 0.8562693931088368, 'content_score': 0.8662839071780959, 'hybrid_score': 0.8602751987365403}, {'rank': 2, 'product_id': 158, 'product_name': 'Product 158', 'category': 'Sports', 'brand': 'Brand 36', 'price': 739.1, 'cf_score': 0.9741745861975637, 'content_score': 0.6527245124540006, 'hybrid_score': 0.8455945567001385}, {'rank': 3, 'product_id': 143, 'product_name': 'Product 143', 'category': 'Books', 'brand': 'Brand 4', 'price': 692.23, 'cf_score': 0.8832015182679113, 'content_score': 0.7653416433376404, 'hybrid_score': 0.8360575682958029}, {'rank': 4, 'product_id': 343, 'product_name': 'Product 343', 'category': 'Beauty', 'brand': 'Brand 30', 'price': 952.08, 'cf_score': 0.7806589130277285, 'content_score': 0.9025497394597313, 'hybr

In [39]:
response = requests.get(
    "http://127.0.0.1:8000/cold-start",
    params={
        "preferred_category": "Electronics",
        "top_n": 5
    }
)

print("Status Code:", response.status_code)
print(response.json())

INFO:     127.0.0.1:47952 - "GET /cold-start?preferred_category=Electronics&top_n=5 HTTP/1.1" 200 OK
Status Code: 200
{'preferred_category': 'Electronics', 'recommendations': [{'rank': 1, 'product_id': 259, 'product_name': 'Product 259', 'category': 'Electronics', 'brand': 'Brand 33', 'price': 112.09, 'total_strength': 163}, {'rank': 2, 'product_id': 565, 'product_name': 'Product 565', 'category': 'Electronics', 'brand': 'Brand 22', 'price': 936.43, 'total_strength': 146}, {'rank': 3, 'product_id': 392, 'product_name': 'Product 392', 'category': 'Electronics', 'brand': 'Brand 11', 'price': 944.35, 'total_strength': 145}, {'rank': 4, 'product_id': 222, 'product_name': 'Product 222', 'category': 'Electronics', 'brand': 'Brand 34', 'price': 768.01, 'total_strength': 141}, {'rank': 5, 'product_id': 55, 'product_name': 'Product 55', 'category': 'Electronics', 'brand': 'Brand 28', 'price': 880.73, 'total_strength': 140}]}


In [40]:
import requests
import pandas as pd
import gradio as gr

def recommendation_ui(customer_id, category):

    # Existing user
    if customer_id is not None and str(customer_id).strip() != "":
        try:
            customer_id = int(customer_id)

            # Gradio → FastAPI
            response = requests.get(
                f"http://127.0.0.1:8000/recommend/{customer_id}",
                params={"top_n": 10}
            )

            if response.status_code != 200:
                return "API Error", pd.DataFrame()

            result = response.json()

            if result["recommendations"]:
                df = pd.DataFrame(result["recommendations"])
                return "Existing User - Hybrid Recommendations", df

        except (ValueError, TypeError):
            return "Invalid Customer ID", pd.DataFrame()

    # New user / Cold Start
    response = requests.get(
        "http://127.0.0.1:8000/cold-start",
        params={
            "preferred_category": category,
            "top_n": 10
        }
    )

    if response.status_code != 200:
        return "API Error", pd.DataFrame()

    result = response.json()

    df = pd.DataFrame(result["recommendations"])

    return "New User - Cold Start Recommendations", df


demo = gr.Interface(
    fn=recommendation_ui,
    inputs=[
        gr.Textbox(
            label="Customer ID",
            placeholder="Example: 1 (leave empty for new user)"
        ),
        gr.Dropdown(
            choices=[
                "Electronics",
                "Sports",
                "Home",
                "Fashion",
                "Books",
                "Beauty"
            ],
            value="Electronics",
            label="Preferred Category"
        )
    ],
    outputs=[
        gr.Textbox(label="Recommendation Type"),
        gr.Dataframe(label="Recommended Products")
    ],
    title=" Intelligent Product Recommendation System",
    description="Hybrid recommendation system using Collaborative Filtering + Content-Based Filtering"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://81dcf57cf412acacf0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [41]:
requirements = """
pandas
numpy
scikit-learn
scipy
scikit-surprise
fastapi
uvicorn
gradio
requests
nest_asyncio
"""

with open("/content/requirements.txt", "w") as f:
    f.write(requirements.strip())

print("requirements.txt created successfully!")

requirements.txt created successfully!


In [42]:
readme_content = """
# Intelligent Product Recommendation System

An AI-powered product recommendation system that combines
Collaborative Filtering and Content-Based Filtering to generate
personalized Top-N product recommendations.

## Project Overview

This project was developed as an AI/ML recommendation system using
user-product interaction data, product attributes, and customer
information.

The system supports:
- Existing-user recommendations
- Cold-start recommendations for new users
- Hybrid recommendation using Collaborative Filtering and
  Content-Based Filtering
- FastAPI backend
- Gradio user interface
"""

with open("/content/README.md", "w") as f:
    f.write(readme_content.strip())

print("README.md created successfully!")

README.md created successfully!


In [43]:
readme_content = """
# Intelligent Product Recommendation System

An AI-powered product recommendation system that combines
Collaborative Filtering and Content-Based Filtering to generate
personalized Top-N product recommendations.

## 1. Project Overview

The system recommends products based on customer interaction
history and product characteristics.

It supports both existing users and new users through a
cold-start recommendation strategy.

## 2. Problem Statement

E-commerce platforms need to recommend relevant products to users
based on their previous interactions and product characteristics.

This project builds a recommendation system that uses multiple
recommendation techniques and combines them using a hybrid approach.

## 3. Dataset

The project uses a synthetic e-commerce dataset containing:

- 1,500 customers
- 600 products
- 30,000 interactions

Main datasets:

- customers.csv
- products.csv
- interactions.csv

The pre-generated recommendations.csv file was not used for
building the recommendation models.

## 4. Data Preprocessing

The following preprocessing steps were performed:

- Checked missing values
- Checked duplicate records
- Validated customer and product IDs
- Converted interaction dates into datetime format
- Created interaction-strength scores

Interaction weights:

- View = 1
- Click = 2
- Wishlist = 3
- Cart = 4
- Purchase = 5

## 5. Recommendation Approaches

### Popularity Baseline

Products were ranked according to their total interaction strength.
This provides a baseline for comparing recommendation models.

### Collaborative Filtering

Item-based Collaborative Filtering was implemented using
user-product interaction data and cosine similarity.

### Content-Based Filtering

Product characteristics such as category, brand, and price were
converted into numerical features.

Cosine similarity was then used to identify similar products.

### SVD

SVD-based matrix factorization was implemented as an additional
Collaborative Filtering approach.

### Hybrid Recommendation

Collaborative Filtering and Content-Based Filtering were combined
using a weighted scoring approach.

Final Score:

Hybrid Score = 0.6 × Collaborative Score + 0.4 × Content Score

The scores were normalized before combining them.

## 6. Cold Start Handling

For new users with no interaction history, the system uses the
user's preferred product category and product popularity to generate
recommendations.

Example:

New user + Electronics preference → popular Electronics products.

## 7. Evaluation

The recommendation models were evaluated using a train/test strategy
where the latest interaction of each customer was held out as the
test interaction.

Evaluation metrics:

- Precision@10
- Recall@10
- F1@10
- NDCG@10

Models evaluated:

- Popularity Baseline
- Collaborative Filtering
- Content-Based Filtering
- SVD
- Hybrid Recommendation

## 8. API

FastAPI was used to provide recommendation endpoints.

### Existing User

GET:

/recommend/{customer_id}

Example:

/recommend/1?top_n=10

### Cold Start

GET:

/cold-start

Example:

/cold-start?preferred_category=Electronics&top_n=10

## 9. User Interface

A Gradio-based user interface was developed.

The UI supports:

- Existing customer recommendations
- New-user cold-start recommendations
- Product recommendation display

Architecture:

Gradio UI → FastAPI → Recommendation Model → FastAPI → Gradio UI

## 10. Testing

The FastAPI endpoints were tested using HTTP requests.

Existing-user recommendation:

Status Code: 200

Cold-start recommendation:

Status Code: 200

The Gradio UI was also tested for existing users and new users.

## 11. Technologies Used

- Python
- Pandas
- NumPy
- Scikit-learn
- SciPy
- Scikit-Surprise
- FastAPI
- Uvicorn
- Gradio

## 12. Project Status

The recommendation models, evaluation, cold-start handling,
FastAPI backend, and Gradio user interface have been implemented
and tested.

## 13. Future Improvements

- Deploy the API and UI to a cloud platform
- Add more advanced personalization
- Improve recommendation diversity
- Add user authentication
- Experiment with additional hybrid weighting strategies
"""

with open("/content/README.md", "w") as f:
    f.write(readme_content.strip())

print("README.md updated successfully!")

README.md updated successfully!


In [44]:
evaluation_results = """
## 14. Evaluation Results

The models were evaluated using Precision@10, Recall@10 and NDCG@10.

| Model | Precision@10 | Recall@10 | NDCG@10 |
|---|---:|---:|---:|
| Popularity Baseline | 0.0018 | 0.0180 | 0.0083 |
| Collaborative Filtering | 0.0019 | 0.0193 | 0.0090 |
| Content-Based Filtering | 0.0013 | 0.0133 | 0.0070 |
| SVD | 0.0013 | 0.0127 | 0.0057 |
| Hybrid Recommendation | 0.0021 | 0.0213 | 0.0100 |

The evaluation used 1,500 customers with one held-out interaction
per customer as the test target.

The Hybrid model combines Collaborative Filtering and
Content-Based Filtering using a 0.6 / 0.4 weighting strategy.
"""

with open("/content/README.md", "a") as f:
    f.write("\n\n" + evaluation_results.strip())

print("Evaluation results added to README.md successfully!")

Evaluation results added to README.md successfully!


In [45]:
architecture = """
## 15. System Architecture

The system follows a simple client-server recommendation architecture.

### Recommendation Flow

1. The user enters a Customer ID or preferred category in the Gradio UI.
2. Gradio sends an HTTP request to the FastAPI backend.
3. FastAPI calls the appropriate recommendation function.
4. The recommendation system generates product recommendations.
5. For existing users, the Hybrid model combines Collaborative Filtering
   and Content-Based Filtering.
6. For new users, the Cold Start strategy uses the preferred category
   and product popularity.
7. FastAPI returns the recommendations as a JSON response.
8. Gradio displays the recommended products to the user.

### Architecture

Gradio UI
    ↓
FastAPI Backend
    ↓
Recommendation Model
    ↓
Hybrid Recommendation / Cold Start
    ↓
FastAPI JSON Response
    ↓
Gradio UI
"""

with open("/content/README.md", "a") as f:
    f.write("\n\n" + architecture.strip())

print("System architecture added to README.md successfully!")

System architecture added to README.md successfully!


In [46]:
import os

screenshots_path = "/content/screenshots"
os.makedirs(screenshots_path, exist_ok=True)

print("screenshots folder created successfully!")

screenshots folder created successfully!


In [47]:
from google.colab import files

uploaded = files.upload()

Saving output.zip to output.zip


In [48]:
import zipfile
import os

with zipfile.ZipFile("/content/output.zip", "r") as zip_ref:
    zip_ref.extractall("/content/screenshots")

print("Screenshots extracted successfully!")
print(os.listdir("/content/screenshots"))

Screenshots extracted successfully!
['output']


In [49]:
import os

output_folder = "/content/screenshots/output"

print(os.listdir(output_folder))

['cold_start_ui.png.png', 'hybrid_recommendations_ui.png.png', 'evaluation_results.png.png', 'hybrid_recommendation_results.png.png']


In [50]:
import os

folder = "/content/screenshots/output"

rename_map = {
    "cold_start_ui.png.png": "cold_start_ui.png",
    "hybrid_recommendations_ui.png.png": "hybrid_recommendations_ui.png",
    "evaluation_results.png.png": "evaluation_results.png",
    "hybrid_recommendation_results.png.png": "hybrid_recommendation_results.png"
}

for old_name, new_name in rename_map.items():
    old_path = os.path.join(folder, old_name)
    new_path = os.path.join(folder, new_name)
    os.rename(old_path, new_path)

print("Screenshot filenames cleaned successfully!")
print(os.listdir(folder))

Screenshot filenames cleaned successfully!
['hybrid_recommendations_ui.png', 'cold_start_ui.png', 'hybrid_recommendation_results.png', 'evaluation_results.png']


In [51]:
import os
import shutil

source_folder = "/content/screenshots/output"
target_folder = "/content/screenshots"

for filename in os.listdir(source_folder):
    source_path = os.path.join(source_folder, filename)
    target_path = os.path.join(target_folder, filename)

    shutil.move(source_path, target_path)

os.rmdir(source_folder)

print("Screenshots moved successfully!")
print(os.listdir(target_folder))

Screenshots moved successfully!
['hybrid_recommendations_ui.png', 'cold_start_ui.png', 'hybrid_recommendation_results.png', 'evaluation_results.png']


In [52]:
screenshots_section = """
## 16. Screenshots

### Hybrid Recommendation UI

![Hybrid Recommendation UI](screenshots/hybrid_recommendations_ui.png)

### Hybrid Recommendation Results

![Hybrid Recommendation Results](screenshots/hybrid_recommendation_results.png)

### Cold Start Recommendation

![Cold Start Recommendation](screenshots/cold_start_ui.png)

### Evaluation Results

![Evaluation Results](screenshots/evaluation_results.png)
"""

with open("/content/README.md", "a") as f:
    f.write("\n\n" + screenshots_section.strip())

print("Screenshots section added to README.md successfully!")

Screenshots section added to README.md successfully!


In [54]:
project_structure = """
## 17. Project Structure

intelligent-product-recommendation/
|
|-- data/
|   |-- customers.csv
|   |-- products.csv
|   |-- interactions.csv
|
|-- notebooks/
|   |-- recommendation_system.ipynb
|
|-- src/
|   |-- preprocessing.py
|   |-- recommendation.py
|   |-- evaluation.py
|   |-- api.py
|
|-- app/
|   |-- ui.py
|
|-- screenshots/
|   |-- hybrid_recommendations_ui.png
|   |-- hybrid_recommendation_results.png
|   |-- cold_start_ui.png
|   |-- evaluation_results.png
|
|-- requirements.txt
|-- README.md
|-- .gitignore
"""

with open("/content/README.md", "a") as f:
    f.write("\\n\\n" + project_structure.strip())

print("Project structure added to README.md successfully!")

Project structure added to README.md successfully!


In [55]:
setup_section = """
## 18. Installation

Clone the repository:

git clone <your-github-repository-url>

Install the required dependencies:

pip install -r requirements.txt

## 19. How to Run

### Step 1: Start the FastAPI Backend

Run the FastAPI application:

uvicorn src.api:app --host 0.0.0.0 --port 8000

### Step 2: Start the Gradio UI

Run the Gradio application:

python app/ui.py

### Step 3: Use the Application

For an existing user:

1. Enter a valid Customer ID.
2. Click Submit.
3. The system sends a request to the FastAPI backend.
4. The Hybrid Recommendation model generates Top-N recommendations.
5. The recommendations are displayed in the Gradio UI.

For a new user:

1. Leave Customer ID empty.
2. Select a preferred category.
3. Click Submit.
4. The Cold Start recommendation strategy generates popular products
   from the selected category.
5. The recommendations are displayed in the Gradio UI.
"""

with open("/content/README.md", "a") as f:
    f.write("\n\n" + setup_section.strip())

print("Installation and run instructions added successfully!")

Installation and run instructions added successfully!


In [56]:
limitations_section = """
## 20. Limitations

- The dataset is a synthetic e-commerce dataset.
- The current evaluation uses one held-out interaction per customer.
- The recommendation quality depends on the available interaction history.
- New users without a preferred category have limited personalization.
- The current system is demonstrated locally using Google Colab.
- The API and UI are not deployed to a production cloud environment.

## 21. Future Improvements

- Deploy FastAPI and Gradio to a cloud platform.
- Use a larger real-world e-commerce dataset.
- Add recommendation diversity and novelty.
- Add more user and contextual features.
- Optimize hybrid weights using a separate validation set.
- Explore advanced recommendation and ranking techniques.
- Add user authentication and persistent user profiles.
"""

with open("/content/README.md", "a") as f:
    f.write("\n\n" + limitations_section.strip())

print("Limitations and future improvements added successfully!")

Limitations and future improvements added successfully!


In [57]:
with open("/content/README.md", "r") as f:
    readme = f.read()

print(readme)

# Intelligent Product Recommendation System

An AI-powered product recommendation system that combines
Collaborative Filtering and Content-Based Filtering to generate
personalized Top-N product recommendations.

## 1. Project Overview

The system recommends products based on customer interaction
history and product characteristics.

It supports both existing users and new users through a
cold-start recommendation strategy.

## 2. Problem Statement

E-commerce platforms need to recommend relevant products to users
based on their previous interactions and product characteristics.

This project builds a recommendation system that uses multiple
recommendation techniques and combines them using a hybrid approach.

## 3. Dataset

The project uses a synthetic e-commerce dataset containing:

- 1,500 customers
- 600 products
- 30,000 interactions

Main datasets:

- customers.csv
- products.csv
- interactions.csv

The pre-generated recommendations.csv file was not used for
building the recommend

In [58]:
with open("/content/README.md", "r") as f:
    readme = f.read()

# Remove duplicate Future Improvements section (Section 13)
start = readme.find("## 13. Future Improvements")
end = readme.find("## 14. Evaluation Results")

if start != -1 and end != -1:
    readme = readme[:start] + readme[end:]

# Fix the visible \n\n text
readme = readme.replace(
    "![Evaluation Results](screenshots/evaluation_results.png)\\n\\n## 17. Project Structure",
    "![Evaluation Results](screenshots/evaluation_results.png)\n\n## 17. Project Structure"
)

with open("/content/README.md", "w") as f:
    f.write(readme)

print("README cleaned successfully!")

README cleaned successfully!


In [59]:
import os

project_path = "/content/intelligent-product-recommendation"

folders = [
    "data",
    "notebooks",
    "src",
    "app",
    "screenshots"
]

for folder in folders:
    os.makedirs(os.path.join(project_path, folder), exist_ok=True)

print("Project folders created successfully!")
print(os.listdir(project_path))

Project folders created successfully!
['screenshots', 'src', 'app', 'notebooks', 'data']


In [60]:
import os
import shutil

project_path = "/content/intelligent-product-recommendation"

# Copy README
shutil.copy(
    "/content/README.md",
    os.path.join(project_path, "README.md")
)

# Copy requirements
shutil.copy(
    "/content/requirements.txt",
    os.path.join(project_path, "requirements.txt")
)

# Copy screenshots
source_screenshots = "/content/screenshots"
target_screenshots = os.path.join(project_path, "screenshots")

for filename in os.listdir(source_screenshots):
    source = os.path.join(source_screenshots, filename)
    target = os.path.join(target_screenshots, filename)

    if os.path.isfile(source):
        shutil.copy(source, target)

print("README, requirements and screenshots copied successfully!")
print(os.listdir(project_path))

README, requirements and screenshots copied successfully!
['README.md', 'screenshots', 'src', 'app', 'notebooks', 'requirements.txt', 'data']


In [62]:
import os

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for file in files:
        if file.endswith(".ipynb"):
            matches.append(os.path.join(root, file))

print("Notebook files found:")
for file in matches:
    print(file)

Notebook files found:
/content/drive/MyDrive/Traffic Sign Detection YOLOV9.ipynb
/content/drive/MyDrive/Untitled0.ipynb
/content/drive/MyDrive/Untitled1.ipynb
/content/drive/MyDrive/intelligent_product_recommendation.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of download_datasets.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of recomending.ipynb
/content/drive/MyDrive/Colab Notebooks/Copy of intelligent_product_recommendation.ipynb
